# 01 · Quickstart

Load the shipped LightGBM demo world-model and run a prediction for a single scenario. No GPU, no LLM, no PyTorch — this notebook runs on any laptop in under a second.

**Prereq**: `pip install -e .` from the repo root.

In [ ]:
import pickle
import numpy as np
import lightgbm as lgb
from pathlib import Path

# 1. Load the shipped pretrained pkl
pkl_path = Path('../data/models/world_model_demo.pkl')
with open(pkl_path, 'rb') as f:
    blob = pickle.load(f)

print(f"model feature_version: {blob['config']['feature_version']}")
print(f"KPIs: {blob['config']['kpis']}")
print(f"quantiles: {blob['config']['quantiles']}")
print(f"n_train: {blob['config']['n_train']}  n_val: {blob['config']['n_val']}")

In [ ]:
# 2. Define a scenario
niches = blob['config']['niches']
tiers = blob['config']['kol_tiers']

scenario = {
    'platform':            'xhs',
    'niche':               'beauty',
    'budget':              80_000.0,
    'budget_bucket':       2,
    'kol_tier':            'mid',
    'kol_fan_count':       240_000,
    'kol_engagement_rate': 0.042,
}

# Feature vector: [platform, niche_idx, budget, bucket, tier_idx, fans, er]
x = np.asarray([[
    0.0,
    float(niches.index(scenario['niche'])),
    scenario['budget'],
    float(scenario['budget_bucket']),
    float(tiers.index(scenario['kol_tier'])),
    float(scenario['kol_fan_count']),
    scenario['kol_engagement_rate'],
]], dtype=np.float32)
x.shape

In [ ]:
# 3. Predict with quantile bands
boosters = {
    kpi: {float(q): lgb.Booster(model_str=s) for q, s in per_q.items()}
    for kpi, per_q in blob['boosters'].items()
}

print(f"{'KPI':<14} {'P35':>10} {'P50':>10} {'P65':>10}")
print('-' * 46)
for kpi in ('impressions', 'clicks', 'conversions', 'revenue'):
    p35 = boosters[kpi][0.35].predict(x)[0]
    p50 = boosters[kpi][0.50].predict(x)[0]
    p65 = boosters[kpi][0.65].predict(x)[0]
    print(f"{kpi:<14} {p35:>10.1f} {p50:>10.1f} {p65:>10.1f}")

## Next

- [02_counterfactual.ipynb](02_counterfactual.ipynb) — run a `do()` intervention and compare against the factual prediction.
- [03_custom_platform.ipynb](03_custom_platform.ipynb) — write your own `PlatformAdapter` for a platform Oransim doesn't ship.
- [04_soul_agents.ipynb](04_soul_agents.ipynb) — have the LLM soul agents react to the creative in natural language.